In [4]:
# If needed, uncomment and run to install requirements
# %pip install --upgrade metpy s3fs fsspec arm_pyart numpy siphon

from datetime import timezone, timedelta, datetime
from pathlib import Path
import os
import shutil

import fsspec
from metpy.io import Level2File, Level3File
from metpy.remote import NEXRADLevel2Archive, NEXRADLevel3Archive

# Optional Level II fallback parser
try:
    import pyart
    HAVE_PYART = True
except Exception:
    pyart = None
    HAVE_PYART = False

# Optional THREDDS fallback for Level III
try:
    from siphon.radarserver import RadarServer
    HAVE_SIPHON = True
except Exception:
    HAVE_SIPHON = False

# ---------------- Config ----------------
SITE_L2 = "KMLB"           # Level II uses 4-letter site IDs
SITE_L3 = "MLB"            # Level III commonly uses 3-letter site IDs
L3_PRODUCTS = ["N0Q"]      # add others like "N0U" as needed
LOOKBACK_HOURS = 4         # past 4 hours
SAVE_DIR = Path("./radar_downloads")
VERIFY_FIRST_ONLY = True   # quick sanity check without parsing everything
# ---------------------------------------

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def save_stream_to_file(fp, out_path: Path) -> None:
    with open(out_path, "wb") as f:
        shutil.copyfileobj(fp, f)

def l2_s3_key_from_name(name: str) -> str:
    # Example name: KMLB20251030_080645_V06
    site = name[:4]
    y, m, d = name[4:8], name[8:10], name[10:12]
    return f"{y}/{m}/{d}/{site}/{name}"

def open_url(url: str):
    """Context manager for s3:// and https:// using fsspec."""
    if url.startswith("s3://"):
        fs_s3 = fsspec.filesystem("s3", anon=True)
        return fs_s3.open(url, "rb")
    return fsspec.open(url, "rb")

def iter_level2(site4: str, start: datetime, end: datetime):
    arc2 = NEXRADLevel2Archive()
    for prod in arc2.get_range(site4.upper(), start, end):
        yield prod

def iter_level3_all(site_hint: str, product: str, start: datetime, end: datetime):
    """Yield (descriptor, site_used, product_used) trying MLB/KMLB and N0Q->N0B."""
    arc3 = NEXRADLevel3Archive()
    sites = [site_hint.upper()]
    if len(site_hint) == 3:
        sites.append("K" + site_hint.upper())
    elif len(site_hint) == 4 and site_hint.upper().startswith("K"):
        sites.append(site_hint.upper()[1:])
    prods = [product.upper()]
    if product.upper() == "N0Q":
        prods.append("N0B")
    for site in sites:
        for prod in prods:
            for desc in arc3.get_range(site, prod, start, end):
                yield desc, site, prod

# --------------- Main -------------------
ensure_dir(SAVE_DIR)
L2_ROOT = SAVE_DIR / "level2"
L3_ROOT = SAVE_DIR / "level3"
ensure_dir(L2_ROOT)
ensure_dir(L3_ROOT)

now = datetime.now(timezone.utc)
start = now - timedelta(hours=LOOKBACK_HOURS)
end = now

print(f"Search window: {start.isoformat()} to {end.isoformat()} UTC")
print(f"Level II site: {SITE_L2}  Level III site: {SITE_L3}  Products: {L3_PRODUCTS}")

# -------- Level II: download all volumes to ./radar_downloads/level2/<SITE>/ --------
l2_descriptors = list(iter_level2(SITE_L2, start, end))
print(f"Found {len(l2_descriptors)} Level II volumes in window.")

l2_downloaded = 0
for i, d in enumerate(l2_descriptors, 1):
    l2_name = os.path.basename(d.name)
    l2_site = l2_name[:4].upper()
    out_dir = L2_ROOT / l2_site
    ensure_dir(out_dir)

    l2_key = l2_s3_key_from_name(l2_name)
    s3_url = f"s3://unidata-nexrad-level2/{l2_key}"
    out_path = out_dir / l2_name
    if out_path.exists():
        continue
    print(f"[L2 {i}/{len(l2_descriptors)}] {s3_url} -> {out_path}")
    with open_url(s3_url) as src:
        save_stream_to_file(src, out_path)
    l2_downloaded += 1

print(f"Level II saved: {l2_downloaded} new files, {len(l2_descriptors) - l2_downloaded} already present.")

# Optional quick parse sanity check on the newest L2
if l2_descriptors and VERIFY_FIRST_ONLY:
    sample_name = os.path.basename(l2_descriptors[-1].name)
    sample_site = sample_name[:4].upper()
    sample_l2 = L2_ROOT / sample_site / sample_name
    try:
        if HAVE_PYART:
            radar = pyart.io.read_nexrad_archive(str(sample_l2))
            print(f"Verified Level II with Py-ART. Sweeps: {radar.nsweeps} Fields: {list(radar.fields.keys())}")
        else:
            with open(sample_l2, "rb") as f:
                l2 = Level2File(f)
            print(f"Verified Level II with MetPy. Station: {getattr(l2, 'stid', '?')} Time: {getattr(l2, 'dt', '?')}")
    except Exception as e:
        print(f"Level II verify parse failed: {e}")

# -------- Level III: download all frames to ./radar_downloads/level3/<SITE>/ --------
total_l3_downloaded = 0
for prod_code in L3_PRODUCTS:
    l3_descs = list(iter_level3_all(SITE_L3, prod_code, start, end))
    print(f"Product {prod_code}: found {len(l3_descs)} Level III descriptors via archive API.")
    seen = set()
    downloaded = 0

    # Primary path: use descriptor URL if available, else build Unidata S3 path
    for i, (d, site_used, prod_used) in enumerate(l3_descs, 1):
        url = getattr(d, "url", None) or f"s3://unidata-nexrad-level3/{d.name}"
        name = os.path.basename(d.name)
        if name in seen:
            continue
        seen.add(name)

        site_dir = L3_ROOT / site_used
        ensure_dir(site_dir)
        out_path = site_dir / name
        if out_path.exists():
            continue

        print(f"[L3 {prod_code} {i}/{len(l3_descs)}] {url} -> {out_path}")
        try:
            with open_url(url) as src:
                save_stream_to_file(src, out_path)
            downloaded += 1
        except Exception as e:
            print(f"  Archive fetch failed for {name} ({e}).")

    # Fallback: THREDDS RadarServer if archive yielded nothing
    if downloaded == 0 and HAVE_SIPHON:
        print(f"Archive empty for {prod_code}. Trying THREDDS RadarServer...")
        try:
            rs = RadarServer("https://thredds.ucar.edu/thredds/radarServer/nexrad/level3/IDD/")
            q = (rs.query()
                   .stations(SITE_L3.upper())
                   .variables(prod_code.upper())
                   .time_range(start, end))
            cat = rs.get_catalog(q)
            total_ds = len(cat.datasets)
            for j, (ds_name, ds) in enumerate(cat.datasets.items(), 1):
                name = os.path.basename(ds_name)
                site_dir = L3_ROOT / SITE_L3.upper()
                ensure_dir(site_dir)
                out_path = site_dir / name
                if out_path.exists():
                    continue
                with ds.remote_open() as src:
                    save_stream_to_file(src, out_path)
                downloaded += 1
                print(f"[L3 {prod_code} THREDDS {j}/{total_ds}] saved {out_path.name}")
        except Exception as e:
            print(f"  THREDDS fallback failed for {prod_code} ({e}).")

    print(f"Level III {prod_code} saved: {downloaded} new files.")
    total_l3_downloaded += downloaded

# -------- List outputs --------
print("\nDownloaded files:")
for root, dirs, files in os.walk(SAVE_DIR):
    for fname in sorted(files):
        print(Path(root) / fname)


Search window: 2025-10-30T05:04:59.852363+00:00 to 2025-10-30T09:04:59.852363+00:00 UTC
Level II site: KMLB  Level III site: MLB  Products: ['N0Q']
Found 33 Level II volumes in window.
[L2 1/33] s3://unidata-nexrad-level2/2025/10/30/KMLB/KMLB20251030_051113_V06 -> radar_downloads\level2\KMLB\KMLB20251030_051113_V06
[L2 2/33] s3://unidata-nexrad-level2/2025/10/30/KMLB/KMLB20251030_051816_V06 -> radar_downloads\level2\KMLB\KMLB20251030_051816_V06
[L2 3/33] s3://unidata-nexrad-level2/2025/10/30/KMLB/KMLB20251030_052519_V06 -> radar_downloads\level2\KMLB\KMLB20251030_052519_V06
[L2 4/33] s3://unidata-nexrad-level2/2025/10/30/KMLB/KMLB20251030_053222_V06 -> radar_downloads\level2\KMLB\KMLB20251030_053222_V06
[L2 5/33] s3://unidata-nexrad-level2/2025/10/30/KMLB/KMLB20251030_053926_V06 -> radar_downloads\level2\KMLB\KMLB20251030_053926_V06
[L2 6/33] s3://unidata-nexrad-level2/2025/10/30/KMLB/KMLB20251030_054629_V06 -> radar_downloads\level2\KMLB\KMLB20251030_054629_V06
[L2 7/33] s3://unidata-